![DB Academy](https://files.training.databricks.com/binder/prod_main/ai-bi-for-data-analysts-en_us-2.0.2/images/20261005T110950Z/AI BI for Data Analysts/Includes/images/common/db-academy.png)

# Demo - Running Your Dashboard in Production

## Overview

A published dashboard is not a fire-and-forget asset. It has to stay **fresh** (the data keeps flowing) and **fast** (consumers wait less than a few seconds to load it). This demo walks through the production controls you have:
  - how dashboards refresh, 
  - how the **dashboard cache** and **query result cache** keep loads fast, 
  - how to set a **refresh schedule**, and how to **subscribe** stakeholders to snapshots. 
  
We finish with a short checklist so your dashboard is truly production-ready.

## Learning Objectives

By the end of this demo, you will be able to:

1. Explain how dashboards refresh (manual, scheduled, filter changes).
2. Describe the **dashboard cache** and **query result cache** and how they affect performance.
3. **Set a refresh schedule** with the appropriate frequency and timezone, and understand how it warms the cache.
4. **Subscribe** stakeholders to snapshot emails (or Slack/Microsoft Teams).
5. Use a **production-ready checklist** to confirm your dashboard is ready to go live.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select a Serverless SQL Warehouse</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless SQL Warehouse (2X-Small is sufficient)**
  - Select **Compute** > **More** > **SQL Warehouse** > Select a SQL Warehouse you have access to.

**NOTE:** This notebook was **developed and tested using Serverless SQL Warehouse**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

<!-- Prerequisites (red) - swap the previous-demo name per notebook -->
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 16px 20px;
  border-radius: 4px;
  margin: 16px 0;
">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;">Complete the previous demonstrations before starting this one.

This demo continues work on the same <strong>Canada Sales</strong> dashboard.
</p>
</div>
</div>
</div>

<!-- Solution shortcut (blue) - paste this same block into every dashboard demo -->
<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 12px 0;
">
<strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.05em;">Need the finished dashboard?</strong>
<div style="color:#333;">

The <strong>Final Demonstration Dashboard</strong> folder contains a notebook (<code>Create Demonstration Solution Dashboard</code>) that recreates the complete solution dashboard in your workspace. Use it if you want to skip ahead, recover from a mistake, or compare your work against the reference build.

</div>
</div>


1. Run the cell below to set up your default catalog and schema.

In [0]:
%run ./Includes/Classroom-Setup-ca_setup

## B. How Dashboards Refresh Data
<br></br>

<div class="rf-wrap">

<style>
.rf-wrap { width: 100%; display: flex; flex-direction: column; align-items: center; gap: 18px; font-family: sans-serif; }

/* Hero header */
.rf-header {
    width: 100%;
    max-width: 1100px;
    background: linear-gradient(135deg, #fff3e0 0%, #ffe0b2 100%);
    border-radius: 10px;
    padding: 22px 28px;
    display: flex;
    align-items: center;
    gap: 18px;
    box-shadow: 0 2px 10px rgba(230,81,0,0.08);
    border-left: 8px solid #FF9800;
    box-sizing: border-box;
}
.rf-icon {
    width: 54px;
    height: 54px;
    border-radius: 50%;
    background: #FF9800;
    color: #FFFFFF;
    font-size: 28pt;
    font-weight: 700;
    display: flex;
    align-items: center;
    justify-content: center;
    flex-shrink: 0;
    line-height: 1;
}
.rf-header-text { flex: 1; color: #0b2026; }
.rf-header-title { font-size: 22pt; font-weight: 700; color: #e65100; line-height: 1.2; margin-bottom: 4px; }
.rf-header-sub { font-size: 14pt; color: #0b2026; line-height: 1.45; }

/* Click hint */
.rf-hint {
    width: 100%;
    max-width: 1100px;
    text-align: center;
    font-size: 14pt;
    color: #5E7077;
    font-style: italic;
    margin-bottom: -4px;
}
.rf-hint strong { color: #e65100; font-style: normal; font-weight: 700; }

/* Button row */
.rf-row {
    display: block;
    white-space: nowrap;
    font-size: 0;
    width: 100%;
    max-width: 1100px;
    box-sizing: border-box;
}
.rf-box {
    display: inline-block;
    width: 32%;
    margin-right: 2%;
    min-height: 90px;
    background: #F9F7F4;
    border-top: 8px solid transparent;
    border-left: 2px solid transparent;
    border-right: 2px solid transparent;
    border-bottom: 2px solid transparent;
    border-radius: 8px;
    padding: 16px 14px;
    text-align: center;
    box-sizing: border-box;
    cursor: pointer;
    user-select: none;
    vertical-align: top;
    transition: transform 0.12s;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    white-space: normal;
}
.rf-box:last-child { margin-right: 0; }
.rf-box:hover { transform: translateY(-2px); }
.rf-box.active {
    background: #fff;
    border-left-color: var(--pc);
    border-right-color: var(--pc);
    border-bottom-color: var(--pc);
}
.rf-label {
    display: block;
    font-weight: 700;
    font-size: 14pt;
    letter-spacing: 2px;
    text-transform: uppercase;
    line-height: 1.2;
    pointer-events: none;
}
.rf-title {
    display: block;
    font-size: 14pt;
    color: #0b2026;
    line-height: 1.3;
    margin-top: 6px;
    pointer-events: none;
}

/* Detail wrap */
.rf-detail-wrap {
    overflow: hidden;
    max-height: 0;
    opacity: 0;
    transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease;
    margin-top: 0;
    width: 100%;
    max-width: 1100px;
}
.rf-detail-wrap.open {
    max-height: 700px;
    opacity: 1;
    margin-top: 12px;
}
.rf-detail-card {
    background: #F9F7F4;
    border-radius: 10px;
    padding: 22px 26px;
    box-sizing: border-box;
    border-top: 7px solid #ccc;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
}
.rf-detail-title {
    font-size: 17pt;
    font-weight: 700;
    color: #0b2026;
    margin-bottom: 12px;
    line-height: 1.3;
}
.rf-detail-body {
    font-size: 14pt;
    color: #0b2026;
    line-height: 1.55;
}
.rf-detail-body ul { margin: 0; padding-left: 22px; }
.rf-detail-body li { margin-bottom: 8px; }
.rf-detail-body li:last-child { margin-bottom: 0; }

/* Common gotcha note card */
.rf-note {
    width: 100%;
    max-width: 1100px;
    background: #F8F9FC;
    border-left: 6px solid #1B5162;
    border-radius: 8px;
    padding: 18px 24px;
    display: flex;
    align-items: center;
    gap: 16px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    box-sizing: border-box;
}
.rf-note-icon {
    width: 44px;
    height: 44px;
    border-radius: 50%;
    background: #1B5162;
    color: #FFFFFF;
    font-size: 22pt;
    font-weight: 700;
    display: flex;
    align-items: center;
    justify-content: center;
    flex-shrink: 0;
    line-height: 1;
}
.rf-note-body { flex: 1; color: #0b2026; }
.rf-note-eyebrow {
    font-size: 14pt;
    font-weight: 700;
    color: #1B5162;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    margin-bottom: 4px;
}
.rf-note-text { font-size: 14pt; color: #0b2026; line-height: 1.5; }
</style>

<!-- Hero -->
<div class="rf-header">
  <div class="rf-icon">⟳</div>
  <div class="rf-header-text">
    <div class="rf-header-title">Dashboards use a 24-hour cache and refresh results when needed</div>
    <div class="rf-header-sub">When a viewer opens a dashboard, Databricks first checks cached results. <br>Fresh data appears when any of the following occur.</div>
  </div>
</div>

<!-- Click hint -->
<div class="rf-hint">For more detail, <strong>click each refresh trigger below</strong>.</div>

<!-- Trigger buttons -->
<div class="rf-row">
  <span class="rf-box" data-id="0" onclick="selectTrigger(0)" style="--pc:#4299E0; border-top-color:#4299E0;">
    <span class="rf-label" style="color:#4299E0;">Manual refresh</span>
    <span class="rf-title">A viewer clicks the Refresh icon button to rerun the queries</span>
  </span><span class="rf-box" data-id="1" onclick="selectTrigger(1)" style="--pc:#00A972; border-top-color:#00A972;">
    <span class="rf-label" style="color:#00A972;">Scheduled refresh</span>
    <span class="rf-title">A configured schedule refreshes the cache</span>
  </span><span class="rf-box" data-id="2" onclick="selectTrigger(2)" style="--pc:#FF5F46; border-top-color:#FF5F46;">
    <span class="rf-label" style="color:#FF5F46;">Filter change</span>
    <span class="rf-title">A new filter value requires a query rerun</span>
  </span>
</div>

<div class="rf-detail-wrap" id="rf-detail-wrap">
  <div class="rf-detail-card" id="rf-detail-card"></div>
</div>

<!-- Common gotcha note card -->
<div class="rf-note">
  <div class="rf-note-icon">!</div>
  <div class="rf-note-body">
    <div class="rf-note-eyebrow">Common gotchas</div>
    <div class="rf-note-text">
      <ul style="margin: 0; padding-left: 20px;">
        <li style="margin-bottom: 12px;"><strong>Dashboards are cache-backed, not live views.</strong> A viewer may see cached results until Databricks reruns queries or a schedule refreshes the cache.</li>
        <li>If a user opens a dashboard <strong>after 24 hours, Databricks may rerun queries</strong> instead of using the older dashboard cache.</li>
        <li style="margin-bottom: 12px;"><strong>Refreshing the browser page does not guarantee fresh data.</strong> On load, Databricks checks cached results first and reruns queries only when needed.</li>
        <li style="margin-bottom: 12px;"><strong>Changing a filter does not always trigger a new query.</strong> Small datasets can update in the browser, while large datasets may rerun queries if that filter combination is not already cached.</li>
      </ul>
    </div>
  </div>
</div>

</div>

<script>
var TRIGGERS = [
  {
    title: "Manual refresh: someone clicks the refresh icon",
    color: "#4299E0",
    text: "<ul><li>A viewer selects the <strong>Refresh</strong> button (circular arrow, top right).</li><li>Available to both viewers and editors.</li><li><strong>All</strong> queries on the dashboard re-run.</li><li>The <strong>cache</strong> updates with the new results.</li></ul>"
  },
  {
    title: "Scheduled refresh: the schedule you set runs",
    color: "#00A972",
    text: "<ul><li>A <strong>schedule</strong> you set runs at the configured time.</li><li>You define a <strong>recurring schedule</strong> (for example, every 15 minutes or every hour).</li><li>All underlying dataset queries <strong>run on schedule</strong>, even if no one has the dashboard open.</li><li>The <strong>cache is pre-warmed</strong>, so the next viewer sees fresh data instantly.</li></ul>"
  },
  {
    title: "Filter change: a viewer changes a filter value",
    color: "#FF5F46",
    text: "<ul><li>A viewer changes a <strong>filter</strong> value (for example, date range, region, product).</li><li>If the new filter value is <strong>already in the cache</strong>, the dashboard updates quickly.</li><li>If the new filter value is <strong>not cached</strong>, it triggers a new query, which can be slower.</li></ul>"
  }
];

var rfCurrent = null;

function selectTrigger(id) {
  var wrap = document.getElementById('rf-detail-wrap');
  var card = document.getElementById('rf-detail-card');
  var t = TRIGGERS[id];

  document.querySelectorAll('.rf-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id && rfCurrent !== id);
  });

  if (rfCurrent === id) {
    wrap.classList.remove('open');
    rfCurrent = null;
    return;
  }

  rfCurrent = id;

  card.style.borderTopColor = t.color;
  card.innerHTML =
    '<div class="rf-detail-title" style="color:' + t.color + ';">' + t.title + '</div>'
    + '<div class="rf-detail-body">' + t.text + '</div>';

  wrap.classList.add('open');
}
</script>

**NOTE:** For those interested in learning more about caching, a bonus deep dive is included in **Appendix A: Dashboard Caching Overview**.

## C. Set a Refresh Schedule

**Manage scheduled dashboard updates and subscriptions** documentation - [AWS](https://docs.databricks.com/aws/en/dashboards/share/schedule-subscribe) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dashboards/share/schedule-subscribe) | [GCP](https://docs.databricks.com/gcp/en/dashboards/share/schedule-subscribe)

A schedule has two jobs:

  - **Keep data fresh** on a predictable cadence. All SQL logic that defines the dashboard's datasets runs at the scheduled time.

  - **Warm the cache** so the next consumer to load the dashboard hits the populated query result cache instead of waiting for queries to run.


1. Make sure you are viewing the **Published** version of your dashboard.

2. Click the **Schedule** button in the upper-right corner. 
    - If no schedule exists, click **+ Schedule**.

3. Here you can specify the **frequency**, **start time**, and **time zone**. 
    - For this demo, explore the UI. No need to set a schedule.
    - The dialog also accepts a **Cron** expression if you need a custom cadence.


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information
  </strong>
  <div style="color:#333;">

**TIPS:** Align the schedule to your upstream pipelines. If your gold table updates at 3:30 AM, schedule the dashboard for 4:00 AM so viewers open to fresh data with the cache already warmed.

**NOTE:** A dashboard can have multiple schedules. Each schedule has its own subscriber list (covered next).

  </div>
</div>


## D. Update the Source Data, Refresh Your Dashboard, Manage Subscriptions

**Manage scheduled dashboard updates and subscriptions** documentation - [AWS](https://docs.databricks.com/aws/en/dashboards/share/schedule-subscribe) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dashboards/share/schedule-subscribe) | [GCP](https://docs.databricks.com/gcp/en/dashboards/share/schedule-subscribe)

### D1. Insert Records Into Your Source Tables

Below we will insert records into the source tables to simulate the data sources being updated. 

1. First, view your dashboard and confirm some of the current values so you can see these values update (**Clear all filters**).
    - **Canadian Sales Overview page**:
      - `Current Total Sales by Sales Target` - (red) $67.83M
      - `Total Orders` - 15,536
      - `Total Units Sold` - 5.08M 
      - `Consistency flows in the area chart`
    - **Opportunities Pipeline page**:
      - `Total Pipeline Value` - $1.12M
      - `Total Opportunities` - 342
      - `Closed Lost Value` - $122.66k

2. Run the cell below to insert records into the data sources of your dashboard to simulate a data update.

In [0]:
-- ca_orders: 10 new rows on 2024-11-15 - 2024-11-18
-- Adds noticeable revenue across multiple reps,
-- so the sales-by-rep bar chart and counters should visibly shift.
INSERT INTO ca_orders
  (orderid, customerid, productid, orderdate, quantity, orderamt, salesrep)
VALUES
  (32001, 'NR3611', 'BC111d', DATE'2024-11-15', 580, 185420.50, 'Ethan'),
  (32002, 'NF9938', 'Q28R3',  DATE'2024-11-15', 720, 2438950.00, 'Hannah'),
  (32003, 'ZD7787', 'A802a',  DATE'2024-11-15', 410, 132475.80, 'Gowoon'),
  (32004, 'MM0344', 'W8931',  DATE'2024-11-15', 295, 1948620.40, 'Ingrid'),
  (32005, 'RV9064', 'JQ9322', DATE'2024-11-15', 640, 215300.75, 'Sam'),
  (32006, 'RS9813', 'BC111d', DATE'2024-11-15', 510, 1622840.00, 'Shalini'),
  (32007, 'NF7397', 'A802a',  DATE'2024-11-16', 380, 1241575.25, 'Ethan'),
  (32008, 'QI5038', 'Q28R3',  DATE'2024-11-16', 695, 1922350.60, 'Hannah'),
  (32009, 'CS8317', 'W8931',  DATE'2024-11-17', 825, 287910.00, 'Sam'),
  (32010, 'AR4386', 'JQ9322', DATE'2024-11-18', 470, 156280.30, 'Ingrid');


-- ca_opportunities: 10 new rows on 2024-10-05 - 2024-10-07
-- Six of the ten land in "2-Closed Won", so the phase
-- breakdown should visibly tilt toward Closed Won after refresh.
INSERT INTO ca_opportunities
  (opportunityid, customerid, productid, province, `date`, amount, salesrep, phase)
VALUES
  ('CA429001', 'NF7397', 'BC111d', 'ON', DATE'2024-10-05', 128450.00, 'Ingrid',  '2-Closed Won'),
  ('CA429002', 'QI5038', 'A802a',  'ON', DATE'2024-10-05', 422800.50, 'Sam',     '2-Closed Won'),
  ('CA429003', 'CS8317', 'W8931',  'BC', DATE'2024-10-05', 15200.75, 'Ethan',   '1-Active'),
  ('CA429004', 'LM6201', 'BC111d', 'SK', DATE'2024-10-05', 19750.00, 'Hannah',  '0-Prospecting'),
  ('CA429005', 'AS0053', 'EE05x',  'ON', DATE'2024-10-06', 524300.25, 'Sam',     '2-Closed Won'),
  ('CA429006', 'RV9064', 'A802a',  'ON', DATE'2024-10-06', 12850.00, 'Gowoon',  '1-Active'),
  ('CA429007', 'AR4386', 'EE05x',  'BC', DATE'2024-10-06', 617600.80, 'Shalini', '2-Closed Won'),
  ('CA429008', 'ZD7787', 'W8931',  'SK', DATE'2024-10-07', 171400.00, 'Ingrid',  '3-Closed Lost'),
  ('CA429009', 'QS1139', 'A802a',  'ON', DATE'2024-10-07', 261900.50, 'Ethan',   '2-Closed Won'),
  ('CA429010', 'NR3611', 'BC111d', 'ON', DATE'2024-10-07', 146200.25, 'Hannah',  '1-Active');

### D2. View and Refresh Your Dashboard

1. View your **Published** dashboard.
    - **Notice the values are still the same and using the old (cached) results.**

2. Select the **Refresh** icon ![Refresh](https://files.training.databricks.com/binder/prod_main/ai-bi-for-data-analysts-en_us-2.0.2/images/20261005T110950Z/AI BI for Data Analysts/Includes/images/common/refresh-icon.png) to refresh entire dashboard cache.

3. View the dashboard and notice the updated values.

    - **Canadian Sales Overview page**:
      - `Current Total Sales by Sales Target` - (red) $77.98M
      - `Total Orders` - 15,546
      - `Total Units Sold` - 5.09M 
      - `Major uptick for the last records in the area chart`
    - **Opportunities Pipeline page**:
      - `Total Pipeline Value` - $3.45M
      - `Total Opportunities` - 352
      - `Closed Lost Value` - $294.06k


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    EXPAND FOR ADDITIONAL DETAILS
  </strong>
  <div style="color:#333;">

<details>

#### What just happened

- You inserted new rows but the dashboard kept showing old numbers.

- That's section B in action: **dashboards do not auto-refresh** when the underlying data changes.

- Clicking **Refresh** forced both caches to update:
  - **Dashboard cache** re-ran the dataset queries.
  - **Query result cache** picked up the new table state.

- The new rows now show up across every visual.

#### In production, schedule it

- Don't ask consumers to click **Refresh** themselves.

- Configure a **scheduled refresh** (section D) to re-run all dataset queries on a recurring cadence and pre-warm the cache.

- **Pattern:** align the schedule to your upstream pipeline. Gold table updates at 4 AM, schedule the dashboard for 4:30 AM.

- Consumers open to fresh data. The first viewer of the day doesn't pay the cold-cache cost.

</details>

  </div>
</div>


### D3. Subscribe to Snapshot Emails

A **subscription** delivers a recurring **PDF snapshot** of the dashboard every time the schedule executes. 

Good for stakeholders who want the report in their inbox without having to click anything. Slack and Microsoft Teams subscriptions are also supported.

1. On the published dashboard, click **Schedule** in the upper-right corner.

2. Select **Advanced settings**

3. Find **Custom email subject** and **Subscribers**. Explore the options. 
    - **NOTES:** 
      - If you cannot add email subscribers, your workspace admin might have turned off email notifications for your workspace.
      - Before you can subscribe Slack or Microsoft Teams channels, workspace admins must configure notification destinations. 

4. Select **Attachments**.
    - Enables you to add subscriptions to support optional tabular data attachments. 


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    What the email contains
  </strong>
  <div style="color:#333;">

- A **PDF snapshot** of the dashboard. For multi-page dashboards, choose which pages to include in the PDF (and their order) using the **Attachments** settings.
- Optional **CSV, TSV, or Excel attachments** of selected widgets (email subscriptions only)
- A link back to the live dashboard.

**NOTE:** subscribers can forward the email. Treat the snapshot as a **screenshot**, not a secured view. If the data is sensitive, add a confidentiality note to the dashboard header, such as `Proprietary. Do not forward outside the organization.`

  </div>
</div>


## E. Scheduling a Full Pipeline with Lakeflow Jobs (High Level Introduction)
<br></br>
<div class="lf-wrap">

<style>
.lf-wrap { width: 100%; display: flex; flex-direction: column; align-items: center; font-family: sans-serif; }
.lf-card { max-width: 1200px; width: 100%; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 18px 22px; position: relative; box-sizing: border-box; color: #0b2026; }
.lf-card::before { content: ""; position: absolute; top: 0; left: 0; width: 100%; height: 8px; background: #1B5162; border-top-left-radius: 8px; border-top-right-radius: 8px; }
.lf-card-label { font-weight: 700; font-size: 13pt; letter-spacing: 2px; text-transform: uppercase; color: #1B5162; margin-top: 6px; }
.lf-card-title { font-size: 17pt; font-weight: 700; color: #0b2026; margin: 2px 0 6px 0; line-height: 1.3; }
.lf-card-sub { font-size: 14pt; color: #5E7077; line-height: 1.5; margin-bottom: 10px; }
.lf-diagram { background: #FFFFFF; border-radius: 6px; padding: 16px; margin: 10px 0; display: flex; justify-content: center; align-items: center; }
.lf-diagram svg { width: 100%; height: auto; max-width: 1100px; }
.lf-accordion { margin-top: 14px; padding: 14px 18px; border-radius: 6px; background: #F0F7FC; }
.lf-accordion-title { font-size: 15pt; font-weight: 700; color: #0b2026; margin: 0 0 10px 0; }
.lf-step { background: #FFFFFF; border: 1px solid #DDE6EE; border-radius: 6px; margin: 0 0 6px 0; overflow: hidden; }
.lf-step:last-child { margin-bottom: 0; }
.lf-step summary { padding: 12px 16px; cursor: pointer; display: flex; align-items: center; gap: 12px; list-style: none; }
.lf-step summary::-webkit-details-marker { display: none; }
.lf-step-num { width: 28px; height: 28px; border-radius: 50%; background: #1B5162; color: #FFFFFF; display: flex; align-items: center; justify-content: center; font-size: 13pt; font-weight: 700; flex-shrink: 0; }
.lf-step-title { font-size: 14pt; font-weight: 700; color: #0b2026; flex: 1; min-width: 0; }
.lf-step-chev { color: #618794; font-size: 12pt; display: inline-block; transition: transform 0.2s ease; flex-shrink: 0; }
.lf-step[open] .lf-step-chev { transform: rotate(90deg); }
.lf-step-body { padding: 4px 16px 14px 56px; font-size: 14pt; line-height: 1.6; color: #0b2026; overflow-wrap: break-word; word-wrap: break-word; }
.lf-step-body code { background: #EEEDE9; padding: 1px 6px; border-radius: 3px; }
.lf-docs { margin-top: 14px; padding-top: 12px; border-top: 1px solid #EEEDE9; font-size: 14pt; color: #0b2026; }
.lf-docs ul { margin: 6px 0 0 0; padding-left: 22px; line-height: 1.7; }
.lf-docs a { color: #1B5162; font-weight: 700; text-decoration: none; }
.lf-docs a:hover { text-decoration: underline; }
</style>

<div class="lf-card">
<div class="lf-card-label">Lakeflow Jobs</div>
<div class="lf-card-title">Beyond a Dashboard Schedule: Orchestrate the Full Pipeline</div>
<div class="lf-card-sub">Place the dashboard refresh as the final task in a Lakeflow Job DAG. The dashboard runs only after upstream ETL has completed and any quality gates have passed.</div>

<div class="lf-diagram">
<svg viewBox="0 0 960 280" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif">
<defs>
<marker id="lf-pass" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" fill="#618794"/></marker>
<marker id="lf-fail" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" fill="#98102A"/></marker>
</defs>
<rect x="20" y="40" width="160" height="110" rx="8" fill="#FFFFFF" stroke="#EEEDE9" stroke-width="1.5"/>
<rect x="20" y="40" width="160" height="6" fill="#4299E0"/>
<text x="100" y="72" text-anchor="middle" font-size="11" font-weight="700" fill="#4299E0" letter-spacing="1.5">NOTEBOOK</text>
<text x="100" y="100" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">bronze_ingestion</text>
<text x="100" y="124" text-anchor="middle" font-size="11" fill="#5E7077">Ingest raw orders</text>
<rect x="210" y="40" width="160" height="110" rx="8" fill="#FFFFFF" stroke="#EEEDE9" stroke-width="1.5"/>
<rect x="210" y="40" width="160" height="6" fill="#00A972"/>
<text x="290" y="72" text-anchor="middle" font-size="11" font-weight="700" fill="#00A972" letter-spacing="1.5">SDP PIPELINE</text>
<text x="290" y="100" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">silver_gold_etl</text>
<text x="290" y="124" text-anchor="middle" font-size="11" fill="#5E7077">Clean &amp; aggregate</text>
<rect x="400" y="40" width="160" height="110" rx="8" fill="#FFFFFF" stroke="#EEEDE9" stroke-width="1.5"/>
<rect x="400" y="40" width="160" height="6" fill="#FFAB00"/>
<text x="480" y="72" text-anchor="middle" font-size="11" font-weight="700" fill="#FFAB00" letter-spacing="1.5">SQL</text>
<text x="480" y="100" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">final_aggregations</text>
<text x="480" y="124" text-anchor="middle" font-size="11" fill="#5E7077">Build dashboard tables</text>
<rect x="590" y="40" width="160" height="110" rx="8" fill="#FFFFFF" stroke="#EEEDE9" stroke-width="1.5"/>
<rect x="590" y="40" width="160" height="6" fill="#618794"/>
<text x="670" y="72" text-anchor="middle" font-size="11" font-weight="700" fill="#618794" letter-spacing="1.5">IF/ELSE</text>
<text x="670" y="100" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">quality_gate</text>
<text x="670" y="124" text-anchor="middle" font-size="11" fill="#5E7077">Row count &gt; threshold?</text>
<rect x="780" y="40" width="160" height="110" rx="8" fill="#FFFFFF" stroke="#EEEDE9" stroke-width="1.5"/>
<rect x="780" y="40" width="160" height="6" fill="#FF5F46"/>
<text x="860" y="72" text-anchor="middle" font-size="11" font-weight="700" fill="#FF5F46" letter-spacing="1.5">DASHBOARDS</text>
<text x="860" y="100" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">refresh_canada_sales</text>
<text x="860" y="124" text-anchor="middle" font-size="11" fill="#5E7077">AI/BI dashboard refresh</text>
<rect x="590" y="190" width="160" height="80" rx="8" fill="#FFFFFF" stroke="#EEEDE9" stroke-width="1.5"/>
<rect x="590" y="190" width="160" height="6" fill="#4299E0"/>
<text x="670" y="220" text-anchor="middle" font-size="11" font-weight="700" fill="#4299E0" letter-spacing="1.5">NOTEBOOK</text>
<text x="670" y="246" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">notify_on_fail</text>
<line x1="180" y1="95" x2="208" y2="95" stroke="#618794" stroke-width="2" marker-end="url(#lf-pass)"/>
<line x1="370" y1="95" x2="398" y2="95" stroke="#618794" stroke-width="2" marker-end="url(#lf-pass)"/>
<line x1="560" y1="95" x2="588" y2="95" stroke="#618794" stroke-width="2" marker-end="url(#lf-pass)"/>
<line x1="750" y1="95" x2="778" y2="95" stroke="#618794" stroke-width="2" marker-end="url(#lf-pass)"/>
<text x="764" y="86" text-anchor="middle" font-size="10" fill="#618794" font-weight="700">pass</text>
<line x1="670" y1="150" x2="670" y2="188" stroke="#98102A" stroke-width="2" marker-end="url(#lf-fail)"/>
<text x="690" y="172" font-size="10" fill="#98102A" font-weight="700">fail</text>
</svg>
</div>

<div class="lf-accordion">
<div class="lf-accordion-title">Why schedule with Lakeflow Jobs instead of just a dashboard schedule?</div>
<details class="lf-step"><summary><span class="lf-step-num">1</span><span class="lf-step-title">End-to-end orchestration</span><span class="lf-step-chev">▶</span></summary>
<div class="lf-step-body">The dashboard refresh runs only after upstream ETL, materialized views, or streaming tables finish successfully. No more dashboards querying half-updated data, and no more racing upstream jobs.</div>
</details>
<details class="lf-step"><summary><span class="lf-step-num">2</span><span class="lf-step-title">Richer triggers and control flow</span><span class="lf-step-chev">▶</span></summary>
<div class="lf-step-body">Cron schedules, file-arrival or table-update triggers, and control flow tasks (if/else, for-each, fan-out/fan-in) let you tie dashboard refreshes to real business events or data arrivals instead of just a fixed cadence.</div>
</details>
<details class="lf-step"><summary><span class="lf-step-num">3</span><span class="lf-step-title">Unified monitoring and retries</span><span class="lf-step-chev">▶</span></summary>
<div class="lf-step-body">Refreshes become first-class job runs, so you get run history, logs, metrics, retry policies, and notifications in the Lakeflow Jobs UI and via system tables. Slow or failed refreshes are easy to find and fix.</div>
</details>
<details class="lf-step"><summary><span class="lf-step-num">4</span><span class="lf-step-title">Dashboards as code</span><span class="lf-step-chev">▶</span></summary>
<div class="lf-step-body">Manage AI/BI dashboards with Declarative Automation Bundles (DABs) and attach the refresh task in the bundle. Refresh and distribution become part of your CI/CD pipeline, so you can promote the same dashboard plus refresh behavior across dev, staging, and prod.</div>
</details>
<details class="lf-step"><summary><span class="lf-step-num">5</span><span class="lf-step-title">Observability and cost</span><span class="lf-step-chev">▶</span></summary>
<div class="lf-step-body">All activity lands in <code>system.lakeflow</code> tables, so you can build internal dashboards to monitor frequency, runtime, failures, and cost of dashboard refresh workloads with the same tools you use for any other job.</div>
</details>
</div>

<div class="lf-docs">
<strong>Documentation</strong>
<ul>
<li>Lakeflow Jobs overview: <a href="https://docs.databricks.com/aws/en/jobs/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/jobs/" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/jobs/" target="_blank">GCP</a></li>
<li>Dashboard task: <a href="https://docs.databricks.com/aws/en/jobs/dashboard" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/jobs/dashboard" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/jobs/dashboard" target="_blank">GCP</a></li>
<li>Configure a task: <a href="https://docs.databricks.com/aws/en/jobs/configure-task" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/jobs/configure-task" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/jobs/configure-task" target="_blank">GCP</a></li>
<li>Triggers: <a href="https://docs.databricks.com/aws/en/jobs/triggers" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/jobs/triggers" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/jobs/triggers" target="_blank">GCP</a></li>
</ul>
</div>

</div>

</div>


1. In the main navigation bar right click on **Jobs & Pipelines** and select **Open Link in New Tab**.

2. Select **Create** > **Job**.

3. Here you can add tasks, set pipline run triggers, refresh the dashboard and more!

For a deep dive training on Lakeflow Jobs, check out the [Deploy Workloads with Lakeflow Jobs](https://www.databricks.com/training/catalog/deploy-workloads-with-lakeflow-jobs-2978) course.

## F. Production-Ready Checklist

**Publish and share a dashboard** documentation - [AWS](https://docs.databricks.com/aws/en/dashboards/share/share) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dashboards/share/share) | [GCP](https://docs.databricks.com/gcp/en/dashboards/share/share)

<br></br>

<div class="pr-wrap">

<style>
.pr-wrap { width: 100%; display: flex; justify-content: center; font-family: sans-serif; }
.pr-card { max-width: 1000px; width: 100%; background: #F9F7F4; border-radius: 10px; padding: 22px 28px; box-sizing: border-box; box-shadow: 0 2px 10px rgba(27,49,57,0.08); border-top: 8px solid #00A972; }
.pr-title { font-size: 18pt; font-weight: 700; color: #0b2026; margin-bottom: 6px; }
.pr-sub { font-size: 13pt; color: #5E7077; margin-bottom: 14px; }
.pr-list { list-style: none; padding: 0; margin: 0; font-size: 14pt; line-height: 1.6; }
.pr-list li { padding: 6px 0 6px 32px; position: relative; color: #0b2026; }
.pr-list li::before { content: "\2713"; position: absolute; left: 4px; top: 6px; color: #00A972; font-weight: 700; font-size: 16pt; }
.pr-list li strong { color: #0b2026; }
</style>

<div class="pr-card">
  <div class="pr-title">Before you announce the dashboard</div>
  <div class="pr-sub">Run through this list once before you hand the URL to stakeholders.</div>
  <ul class="pr-list">
    <li><strong>Published</strong>, not draft.</li>
    <li><strong>Data permission model</strong> chosen intentionally (shared vs individual).</li>
    <li><strong>Sharing permissions</strong> set for the right audience.</li>
    <li><strong>Refresh schedule</strong> configured and aligned with upstream pipeline timing.</li>
    <li><strong>Refresh cadence is communicated</strong> in the dashboard header (best practice).</li>
    <li><strong>Subscribers</strong> added for stakeholders who want snapshot emails.</li>
    <li><strong>Sensitivity note</strong> added if the data is confidential.</li>
    <li><strong>Filters have sensible defaults</strong> (usually "All") so consumers see the full picture on load.</li>
    <li><strong>All widget titles and axis labels are business-friendly</strong>. No raw <code>sum(amount)</code> or <code>monthly(date)</code>.</li>
  </ul>
</div>

</div>

## Conclusion

In this demo, you turned your published dashboard into a production-ready asset:

1. Learned that dashboards **do not auto-refresh**. Manual, scheduled, and filter changes trigger fresh data.
2. Saw how the **dashboard cache** and **query result cache** work together, and why a scheduled refresh on a shared-credential dashboard is the biggest performance win.
3. **Set a refresh schedule** with a timezone, aligned it with upstream pipelines, and communicated the cadence in the header.
4. **Subscribed** yourself and stakeholders to PDF snapshot emails.
5. Walked through the **production-ready checklist** before announcing the dashboard.

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>